# 02 — Architecture experiments

Train models with the same pipeline as `ipe-train` (frozen split, scaling, loss, checkpoints,
metrics) and compare them.

* **Registered models** live in `src/ipe_prediction/training/models/`, one file per model with its
  own config dataclass (`cnn_gru.py`, `tcn_attn.py`).
* **New ideas** can be written directly in a cell as a plain `nn.Module`:

| | shape | content |
|---|---|---|
| input | `(B, C, 480)` | `C = 2`: `[ΔT / std, heater_on]`; `C = 5` with `extra_features=True` (adds `t / L`, cumulative heating, integrated rise) |
| output | `(B, 5)` | standardised `[HLC_mit, HLC_ext, log τ_b, log τ_n, log τ_inf]`; `HLC = HLC_mit + HLC_ext` |

Each run is saved in `outputs/runs/<name>/`.

In [ ]:
import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)  # the configuration uses paths relative to the project root

from ipe_prediction.data import TARGET_NAMES
from ipe_prediction.training.config import DataConfig, ExperimentConfig, TrainConfig
from ipe_prediction.training.dataset import prepare_data
from ipe_prediction.training.engine import load_model
from ipe_prediction.training.models.cnn_gru import CNNGRUConfig
from ipe_prediction.training.models.tcn_attn import TCNAttentionConfig
from ipe_prediction.training.preprocessing import N_OUTPUTS, n_inputs
from ipe_prediction.training.train import run

pd.set_option("display.precision", 3)

EPOCHS = 5
MAX_TRAIN = 100_000  # None = the full training split (800k)
data = prepare_data(DataConfig())  # loaded once, reused by every run (~2.4 GB RAM)


def train(name, model="tcn_attn", model_config=None, net=None, extra_features=False, **train_kw):
    # Train one run from scratch (an existing run with this name is deleted first).
    cfg = ExperimentConfig(
        name,
        model=model,
        model_config=model_config,
        data=DataConfig(max_train=MAX_TRAIN, extra_features=extra_features),
        train=TrainConfig(epochs=EPOCHS, **train_kw),
    )
    shutil.rmtree(cfg.run_dir, ignore_errors=True)
    return run(cfg, model=net, data=data)

## 1. Registered models

Pick a model by name and change its hyper-parameters through its config dataclass.

In [ ]:
train("nb_tcn", model="tcn_attn", model_config=TCNAttentionConfig(channels=64, heads=4))

In [ ]:
train("nb_gru", model="cnn_gru", model_config=CNNGRUConfig(gru_hidden=64))

## 2. A model defined in the notebook

Any `nn.Module` with the input / output contract above. Pass an *untrained instance* as `net`;
`model` is then only the label saved in the run's config.

In [ ]:
class SimpleCNN(nn.Module):
    # Strided convolutions then an average over time.

    def __init__(self, in_channels: int, width: int = 64) -> None:
        super().__init__()
        layers, c = [], in_channels
        for c_out in (32, width, width, width):
            layers += [nn.Conv1d(c, c_out, 5, stride=2, padding=2), nn.BatchNorm1d(c_out), nn.GELU()]
            c = c_out
        self.conv = nn.Sequential(*layers)  # 480 -> 30 steps
        self.head = nn.Sequential(nn.Linear(c, 64), nn.GELU(), nn.Linear(64, N_OUTPUTS))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.conv(x).mean(dim=2))


train("nb_simple_cnn", model="simple_cnn", net=SimpleCNN(n_inputs(extra_features=False)))

To keep a notebook model, move it to `models/<name>.py` with a config dataclass and register both
in `MODELS` (`models/__init__.py`); it is then available to `ipe-train --model <name>`.

## 3. Comparison

Best validation epoch of each run (loss 1.0 = predicting the training mean) and learning curves.

In [ ]:
RUNS = ["nb_tcn", "nb_gru", "nb_simple_cnn"]
hist = pd.concat([pd.read_csv(f"outputs/runs/{r}/history.csv").assign(run=r) for r in RUNS])

best = hist.sort_values("val_loss").groupby("run").head(1)  # best epoch of each run
mape = [f"val_MAPE_{n}" for n in TARGET_NAMES]
display(best.set_index("run")[["epoch", "val_loss", "train_loss", "seconds", *mape]].sort_values("val_loss"))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, metric in zip(axes, ["train_loss", "val_loss"]):
    for r, h in hist.groupby("run", sort=False):
        ax.plot(h["epoch"], h[metric], marker=".", label=r)
    ax.set(xlabel="epoch", title=metric)
axes[0].legend();

## 4. Reload a run and inspect predictions

Registered models are rebuilt from the run's config; a notebook model needs an untrained
instance (`load_model(run_dir, model=SimpleCNN(2))`).

In [ ]:
model, prep, cfg = load_model("outputs/runs/nb_tcn")
idx = data[1]["val"][:5]
tensors = data[0]
with torch.no_grad():
    pred = prep.decode(model(prep.inputs(tensors.delta_t[idx], tensors.heater_on[idx].float())))
true = tensors.targets[idx]
pd.concat(
    {n: pd.DataFrame({"true": true[:, j], "pred": pred[:, j]}) for j, n in enumerate(TARGET_NAMES)},
    axis=1,
)